In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

PARQUET_PATH = Path("../../data/processed/dvf_final.parquet")
CSV_PATH = Path("../../data/processed/dvf_final_mysql.csv")

DB_USER = "root"
DB_PASSWORD = "stephane-34"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "immobilier_dvf"
TABLE_NAME = "transactions"

print("Lecture du parquet...")
df = pd.read_parquet(PARQUET_PATH)

print("Export CSV...")
df.to_csv(
    CSV_PATH,
    index=False,
    sep=";",
    encoding="utf-8",
    na_rep="\\N"
)

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?local_infile=1"
)

print("Création table...")
df.head(0).to_sql(
    TABLE_NAME,
    con=engine,
    if_exists="replace",
    index=False
)

csv_mysql_path = str(CSV_PATH.resolve()).replace("\\", "/")

print("Import rapide MySQL...")
query = f"""
LOAD DATA LOCAL INFILE '{csv_mysql_path}'
INTO TABLE {TABLE_NAME}
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ';'
OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 LINES;
"""

with engine.begin() as conn:
    conn.execute(text(query))

print("Import terminé ✅")

Lecture du parquet...
Export CSV...
Création table...
Import rapide MySQL...
Import terminé ✅
